## 1. Mount Google Drive



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Cài thư viện cần thiết

Các thư viện này phục vụ cleaning text, emoji tokenization, segmentation hashtag và LightGBM model đã lưu trong bundle.


In [3]:
# Cài thư viện cần cho inference/demo
!pip install ftfy emoji wordsegment lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 82.4 MB/s eta 0:00:00


## 3. Import thư viện



In [4]:
# Import các thư viện dùng trong demo inference
import os
import re
import ftfy
import emoji
import unicodedata
import joblib
import numpy as np
import pandas as pd

from wordsegment import load, segment
from sklearn.pipeline import Pipeline

load()

## 4. Load model bundle

File `.joblib` này đã đóng gói toàn bộ mô hình demo, gồm:

- 3 `fold_bundles`: mỗi fold có `preprocessor`, `selector`, SVM, CNB, LGBM.
- `meta_model`: Logistic Regression tầng stacking.
- `class_list`: thứ tự class.
- `class_decision_weights`: trọng số quyết định cuối.



In [5]:
# sửa MODEL_PATH cho đúng vị trí file .joblib.
MODEL_PATH = "/content/drive/MyDrive/CS114/TeamModel/demo_model/tfidf_stacking_demo_bundle.joblib"

bundle = joblib.load(MODEL_PATH)

print(bundle.keys())
print(bundle["experiment_name"])
print(bundle["class_list"])
print(bundle["class_decision_weights"])

dict_keys(['experiment_name', 'fold_bundles', 'meta_model', 'class_list', 'class_decision_weights', 'class_decision_weights_dict', 'best_val_macro_f1', 'n_splits', 'note'])
tfidf_stack_no_leak_word13_charwb35_k35000_v1
['anxiety' 'depression' 'normal' 'suicidal']
[1.   1.4  1.   1.15]


## 5. Cleaning raw text




In [6]:

def fix_encoding(text: str) -> str:
    text = ftfy.fix_text(str(text)).lower()

    text = re.sub(
        r'(\+?\d{1,3}[\s.-]?)?(\d{3,4}[\s.-]?){2,3}',
        '',
        text
    )

    text = re.sub(
        r'(?m)^\s*\d+\s*[\.\)\-]\s*',
        '',
        text
    )

    return text


def convert_emoji(text: str, language: str = "en") -> str:
    text = emoji.demojize(str(text), language=language)
    text = re.sub(r":([a-zA-Z0-9_+-]+):", r" emoji_\1 ", text)
    return text


def clean_urls(text: str) -> str:
    pattern = r"https?://\S+|www\.\S+"
    return re.sub(pattern, " <url> ", str(text))


def clean_mentions(text: str) -> str:
    return re.sub(r"@\w+", " <username> ", str(text))


def split_camel(text: str) -> str:
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', str(text))
    text = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', text)
    return text


def is_noise_hashtag(body: str) -> bool:
    body = str(body)

    if re.fullmatch(r'[\d\W_]+', body):
        return True

    if len(body) <= 2:
        return True

    if (
        re.fullmatch(r'[a-zA-Z0-9]{3,6}', body)
        and re.search(r'\d', body)
        and re.search(r'[a-zA-Z]', body)
    ):
        return True

    return False


def clean_hashtags(text: str) -> str:
    text = str(text)
    hashtags = re.findall(r'#(\w+)', text)

    for body in hashtags:
        if is_noise_hashtag(body):
            text = re.sub(r'#' + re.escape(body) + r'\b', '', text)
            continue

        camel_split = split_camel(body)
        words = segment(camel_split.lower())
        body_clean = ' '.join(words)

        text = re.sub(r'#' + re.escape(body) + r'\b', body_clean, text)

    return text.strip()


def normalize_repeated_chars(text: str, max_repeat: int = 2) -> str:
    text = str(text)
    pattern = rf"([a-zA-Z])\1{{{max_repeat},}}"
    replacement = r"\1" * max_repeat
    text = re.sub(pattern, replacement, text)
    return re.sub(r'\.{2,}', '...', text)


def normalize_unicode(text: str) -> str:
    text = str(text)

    text = re.sub(r"[\u2018\u2019\u201a\u201b]", "'", text)
    text = re.sub(r"[\u201c\u201d\u201e\u201f]", '"', text)
    text = re.sub(r"[\u2013\u2014\u2015]", "-", text)
    text = re.sub(r"\u2026", "...", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", text)

    return text


def normalize_to_ascii(text: str) -> str:
    normalized = unicodedata.normalize('NFD', str(text))
    ascii_text = normalized.encode('ascii', 'ignore').decode('utf-8')
    return ascii_text


def normalize_whitespace(text: str) -> str:
    text = str(text)
    text = re.sub(r"[\n\t\r]", " ", text)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()


def clean_text(
    text: str,
    emoji_language: str = "en",
    keep_case: bool = True,
) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""

    text = fix_encoding(text)
    text = convert_emoji(text, emoji_language)
    text = clean_urls(text)
    text = clean_mentions(text)
    text = clean_hashtags(text)
    text = normalize_repeated_chars(text)
    text = normalize_unicode(text)
    text = normalize_to_ascii(text)
    text = normalize_whitespace(text)

    if not keep_case:
        text = text.lower()

    return text

## 6. Tạo feature phụ cho model

- `cleaned_text`
- `word_count`
- `char_len`
- `avg_word_len`
- `uppercase_ratio`
- `i_pronoun_ratio`
- `negation_ratio`
- `ellipsis_count`
- `exclamation_count`
- `question_count`
- `suicide_keyword_count`


In [7]:
# Input của hàm nên là text đã preprocess

def extract_advanced_features_for_demo(texts):

    if isinstance(texts, str):
        texts = [texts]

    df_out = pd.DataFrame({
        "text": [str(t) for t in texts]
    })

    raw_text = df_out["text"].astype(str)

    df_out["cleaned_text"] = raw_text.str.lower().str.strip()

    df_out["cleaned_text"] = df_out["cleaned_text"].apply(
        lambda x: re.sub(r"(.)\1{4,}", r"\1\1", x)
    )

    df_out["char_len"] = raw_text.str.len()

    df_out["word_count"] = raw_text.apply(
        lambda x: len(x.split())
    )

    df_out["avg_word_len"] = df_out["char_len"] / (df_out["word_count"] + 1e-5)

    df_out["uppercase_ratio"] = raw_text.apply(
        lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1e-5)
    )

    df_out["i_pronoun_ratio"] = df_out["cleaned_text"].apply(
        lambda x: len(re.findall(r"\b(i|me|my|myself|mine)\b", x)) / (len(x.split()) + 1e-5)
    )

    df_out["negation_ratio"] = df_out["cleaned_text"].apply(
        lambda x: len(re.findall(r"\b(no|not|never|nothing|none|cannot|dont|don't|cant|can't)\b", x)) / (len(x.split()) + 1e-5)
    )

    df_out["ellipsis_count"] = raw_text.apply(
        lambda x: len(re.findall(r"\.\.\.", x))
    )

    df_out["exclamation_count"] = raw_text.apply(
        lambda x: x.count("!")
    )

    df_out["question_count"] = raw_text.apply(
        lambda x: x.count("?")
    )

    df_out["suicide_keyword_count"] = df_out["cleaned_text"].apply(
        lambda x: len(
            re.findall(
                r"\b(suicide|suicidal|kill myself|kms|kys|end my life|want to die|wanna die|die|overdose|hang myself)\b",
                x
            )
        )
    )

    feature_cols = [
        "cleaned_text",
        "word_count",
        "char_len",
        "avg_word_len",
        "uppercase_ratio",
        "i_pronoun_ratio",
        "negation_ratio",
        "ellipsis_count",
        "exclamation_count",
        "question_count",
        "suicide_keyword_count"
    ]

    return df_out[feature_cols]

## 7. Hàm predict chính

Quy trình trong `predict_tfidf_stacking()`:

1. Nhận raw text hoặc list text.
2. Nếu có `clean_fn`, preprocess text trước.
3. Tạo feature phụ bằng `extract_advanced_features_for_demo()`.
4. Mỗi fold model dự đoán ra 12 meta-features.
5. Lấy trung bình meta-features từ 3 fold.
6. Meta Logistic Regression dự đoán xác suất cuối.
7. Nhân `class_decision_weights`.
8. Chọn class có adjusted score cao nhất.


In [8]:
# predict_tfidf_stacking: hàm inference chính dùng cho demo.

# aligned_predict_proba: đảm bảo xác suất các model luôn theo đúng thứ tự class_list.
# get_feature_pipeline: dựng lại pipeline preprocessor -> selector từ fold_bundle.

def aligned_predict_proba(model, X, class_names):
    """
    Đảm bảo xác suất predict_proba đúng thứ tự class_names.
    """
    proba = model.predict_proba(X)
    aligned = np.zeros((X.shape[0], len(class_names)), dtype=np.float32)

    for src_idx, cls in enumerate(model.classes_):
        dst_idx = np.where(class_names == cls)[0][0]
        aligned[:, dst_idx] = proba[:, src_idx]

    return aligned


def get_feature_pipeline(fold_bundle):

    fp = fold_bundle["feature_pipeline"]

    if isinstance(fp, dict):
        return Pipeline([
            ("preprocessor", fp["preprocessor"]),
            ("selector", fp["selector"])
        ])

    if hasattr(fp, "transform"):
        return fp

    raise TypeError(f"feature_pipeline không hợp lệ: {type(fp)}")


def predict_tfidf_stacking(texts, bundle, clean_fn=None):

    if isinstance(texts, str):
        texts = [texts]

    texts = [str(t) for t in texts]

    if clean_fn is not None:
        processed_texts = [clean_fn(t) for t in texts]
    else:
        processed_texts = texts

    input_features = extract_advanced_features_for_demo(processed_texts)

    fold_bundles = bundle["fold_bundles"]
    meta_model = bundle["meta_model"]
    class_list = np.array(bundle["class_list"])
    class_weights = np.array(bundle["class_decision_weights"], dtype=np.float32)

    fold_meta_features = []

    for fold_bundle in fold_bundles:
        feature_pipeline = get_feature_pipeline(fold_bundle)
        models = fold_bundle["models"]

        X_fe = feature_pipeline.transform(input_features)

        svm_proba = aligned_predict_proba(
            models["svm"],
            X_fe,
            class_list
        )

        cnb_proba = aligned_predict_proba(
            models["cnb"],
            X_fe,
            class_list
        )

        lgb_proba = aligned_predict_proba(
            models["lgb"],
            X_fe,
            class_list
        )

        meta_features = np.hstack([
            svm_proba,
            cnb_proba,
            lgb_proba
        ])

        fold_meta_features.append(meta_features)

    meta_X = np.mean(fold_meta_features, axis=0)

    final_proba_raw = meta_model.predict_proba(meta_X)

    final_proba = np.zeros((len(texts), len(class_list)), dtype=np.float32)

    for src_idx, cls in enumerate(meta_model.classes_):
        dst_idx = np.where(class_list == cls)[0][0]
        final_proba[:, dst_idx] = final_proba_raw[:, src_idx]

    adjusted_scores = final_proba * class_weights

    pred_idx = np.argmax(adjusted_scores, axis=1)
    pred_labels = class_list[pred_idx]

    result_df = pd.DataFrame({
        "raw_text": texts,
        "processed_text": processed_texts,
        "predicted_label": pred_labels
    })

    for i, cls in enumerate(class_list):
        result_df[f"prob_{cls}"] = final_proba[:, i]
        result_df[f"adjusted_{cls}"] = adjusted_scores[:, i]

    return result_df

## 8. Demo bằng text người dùng nhập trực tiếp


Sửa danh sách `samples`, sau đó chạy cell để xem kết quả.

Vì `samples` là raw text nên bắt buộc dùng:

```python
clean_fn=clean_text
```


In [9]:
# Danh sách câu demo.
# Vì đây là raw text, phải dùng clean_fn=clean_text.

samples = [
    "I feel so hopeless and empty lately.",
    "I want to end my life.",
    "I feel okay today.",
    "I keep having panic attacks and cannot calm down.",
    "I can't do this anymore 😭😭😭"
]

result = predict_tfidf_stacking(
    samples,
    bundle,
    clean_fn=clean_text
)

display(result)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,raw_text,processed_text,predicted_label,prob_anxiety,adjusted_anxiety,prob_depression,adjusted_depression,prob_normal,adjusted_normal,prob_suicidal,adjusted_suicidal
0,I feel so hopeless and empty lately.,i feel so hopeless and empty lately.,depression,0.073589,0.073589,0.454213,0.635899,0.160901,0.160901,0.311297,0.357992
1,I want to end my life.,i want to end my life.,suicidal,0.003361,0.003361,0.065732,0.092024,0.001426,0.001426,0.929482,1.068904
2,I feel okay today.,i feel okay today.,normal,0.063680,0.063680,0.089146,0.124805,0.729391,0.729391,0.117783,0.135451
3,I keep having panic attacks and cannot calm down.,i keep having panic attacks and cannot calm down.,anxiety,0.435161,0.435161,0.309032,0.432645,0.057040,0.057040,0.198768,0.228583
4,I can't do this anymore 😭😭😭,i can't do this anymore emoji_loudly_crying_fa...,normal,0.092970,0.092970,0.056405,0.078968,0.739796,0.739796,0.110829,0.127453


## 9. Kiểm tra lại trên tập test đã preprocess

Phần này chỉ để kiểm tra Model được load chạy có ổn như trên notebook train không.


In [10]:
# Đọc file test đã preprocess sẵn.
# Dùng phần này như sanity check, không phải phần demo chính.

TEST_PATH = "/content/drive/MyDrive/CS114/TeamModel/data/test_clean_tokenizedEmoji.csv"

test_df = pd.read_csv(TEST_PATH, encoding="utf-8")

test_df = test_df.dropna(subset=["text", "status"]).copy()
test_df["status"] = test_df["status"].astype(str).str.strip().str.lower()

print(test_df.shape)
display(test_df.head())

(9887, 2)


,text,status
0,i don't know why i feel so irrationally. i hav...,anxiety
1,there were a lot of moms protesting quarintine...,normal
2,doesn t want him to go,normal
3,"yes, i can take a day off '",normal
4,people try to be funny with me and i can't thi...,anxiety


In [11]:
# Do dùng tập test đã qua cleaning nên clean_fn=None

test_result = predict_tfidf_stacking(
    test_df["text"].tolist(),
    bundle,
    clean_fn=None
)

display(test_result.head())

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,raw_text,processed_text,predicted_label,prob_anxiety,adjusted_anxiety,prob_depression,adjusted_depression,prob_normal,adjusted_normal,prob_suicidal,adjusted_suicidal
0,i don't know why i feel so irrationally. i hav...,i don't know why i feel so irrationally. i hav...,anxiety,0.983720,0.983720,0.009237,0.012931,0.005754,0.005754,0.001289,0.001483
1,there were a lot of moms protesting quarintine...,there were a lot of moms protesting quarintine...,normal,0.027818,0.027818,0.018596,0.026034,0.926259,0.926259,0.027328,0.031427
2,doesn t want him to go,doesn t want him to go,normal,0.015237,0.015237,0.007158,0.010021,0.966059,0.966059,0.011546,0.013278
3,"yes, i can take a day off '","yes, i can take a day off '",normal,0.016183,0.016183,0.007393,0.010350,0.964200,0.964200,0.012224,0.014058
4,people try to be funny with me and i can't thi...,people try to be funny with me and i can't thi...,anxiety,0.743190,0.743190,0.182258,0.255162,0.044570,0.044570,0.029982,0.034480


In [12]:
# Tính lại metric để kiểm tra model.
# Macro F1 khoảng 0.81 khớp với notebook train

from sklearn.metrics import classification_report, f1_score, accuracy_score

y_true = test_df["status"].values
y_pred = test_result["predicted_label"].values

print(classification_report(y_true, y_pred, digits=4))
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))

              precision    recall  f1-score   support

     anxiety     0.8171    0.8655    0.8406      1115
  depression     0.7496    0.7447    0.7471      2902
      normal     0.9360    0.9314    0.9337      3630
    suicidal     0.7246    0.7152    0.7198      2240

    accuracy                         0.8202      9887
   macro avg     0.8068    0.8142    0.8103      9887
weighted avg     0.8200    0.8202    0.8200      9887

Accuracy: 0.8201678972387985
Macro F1: 0.8103137801913176
